In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

In [ ]:
# Upload your CSV file

from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

df.head()

In [ ]:
# Inspect the dataset

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

In [ ]:
# Try to detect date columns

for col in df.columns:
    if df[col].dtype == "object":
        try:
            converted = pd.to_datetime(df[col], errors="raise")
            df[col] = converted
        except:
            pass

df.info()

In [ ]:
# Automatically detect column types

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
date_cols = df.select_dtypes(include=["datetime64"]).columns.tolist()

column_summary = {
    "number_of_rows": df.shape[0],
    "number_of_columns": df.shape[1],
    "columns": df.columns.tolist(),
    "numeric_columns": numeric_cols,
    "categorical_columns": categorical_cols,
    "date_columns": date_cols
}

column_summary

In [ ]:
# Column classification; avoid analyzing fields like order_id and automatically pick a useful main metric.

def find_columns_by_keywords(columns, keywords):
    matches = []
    for col in columns:
        col_lower = col.lower()
        if any(keyword in col_lower for keyword in keywords):
            matches.append(col)
    return matches


id_keywords = ["id", "code", "number", "zip", "phone", "invoice"]

metric_keywords = [
    "sales", "revenue", "amount", "price", "cost", "profit",
    "quantity", "qty", "units", "score", "rating", "satisfaction",
    "inventory", "return", "discount", "margin", "value", "total"
]

id_like_cols = find_columns_by_keywords(df.columns, id_keywords)

useful_numeric_cols = [
    col for col in numeric_cols
    if col not in id_like_cols
    and df[col].nunique() < len(df) * 0.95
]

business_metric_cols = find_columns_by_keywords(useful_numeric_cols, metric_keywords)

if business_metric_cols:
    primary_metric = business_metric_cols[0]
elif useful_numeric_cols:
    primary_metric = useful_numeric_cols[0]
else:
    primary_metric = None

print("ID-like columns excluded:", id_like_cols)
print("Useful numeric columns:", useful_numeric_cols)
print("Selected primary metric:", primary_metric)

In [ ]:
# Numeric summary

numeric_summary = df[useful_numeric_cols].describe().round(2).to_dict() if useful_numeric_cols else {}

numeric_summary

In [ ]:
# Create generic numeric summaries

numeric_summary = df[numeric_cols].describe().round(2).to_dict() if numeric_cols else {}

numeric_summary

In [ ]:
# Create generic categorical summaries

categorical_summary = {}

for col in categorical_cols[:5]:
    categorical_summary[col] = df[col].value_counts(dropna=False).head(5).to_dict()

categorical_summary

In [ ]:
# Missing value summary

missing_summary = (
    df.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[missing_summary > 0].head(10).to_dict()

missing_summary

In [ ]:
# Anomaly detection - This finds unusually high and unusually low values across useful numeric columns.

anomaly_summary = {}

for col in useful_numeric_cols[:5]:
    mean_value = df[col].mean()
    std_value = df[col].std()

    if pd.notnull(std_value) and std_value > 0:
        high_threshold = mean_value + (2 * std_value)
        low_threshold = mean_value - (2 * std_value)

        high_anomalies = df[df[col] > high_threshold][col].head(5).tolist()
        low_anomalies = df[df[col] < low_threshold][col].head(5).tolist()

        if high_anomalies or low_anomalies:
            anomaly_summary[col] = {
                "mean": round(mean_value, 2),
                "std": round(std_value, 2),
                "high_threshold": round(high_threshold, 2),
                "low_threshold": round(low_threshold, 2),
                "sample_high_values": high_anomalies,
                "sample_low_values": low_anomalies
            }

anomaly_summary

In [ ]:
# Relationship summary - This compares the primary metric across categories using count, average, and total.

relationship_summary = {}

if primary_metric and categorical_cols:
    for cat_col in categorical_cols[:5]:
        grouped = (
            df.groupby(cat_col)[primary_metric]
            .agg(["count", "mean", "sum"])
            .sort_values("sum", ascending=False)
            .head(5)
            .round(2)
        )

        relationship_summary[f"{primary_metric}_by_{cat_col}"] = grouped.to_dict()

relationship_summary

In [ ]:
# Concentration analysis - This checks whether one category is dominating the metric.

concentration_summary = {}

if primary_metric and categorical_cols:
    for cat_col in categorical_cols[:5]:
        grouped = (
            df.groupby(cat_col)[primary_metric]
            .sum()
            .sort_values(ascending=False)
        )

        total = grouped.sum()

        if total != 0 and len(grouped) > 0:
            top_value = grouped.iloc[0]
            top_share = round((top_value / total) * 100, 2)

            concentration_summary[cat_col] = {
                "top_group": grouped.index[0],
                "top_group_value": round(top_value, 2),
                "top_group_share_percent": top_share
            }

concentration_summary

In [ ]:
# Correlation analysis

correlation_summary = {}

if len(useful_numeric_cols) >= 2:
    corr_matrix = df[useful_numeric_cols].corr(numeric_only=True).round(2)

    for col in corr_matrix.columns:
        strong_corrs = (
            corr_matrix[col]
            .drop(labels=[col])
            .abs()
            .sort_values(ascending=False)
            .head(3)
        )

        correlation_summary[col] = strong_corrs.to_dict()

correlation_summary

In [ ]:
# Segment analysis

segment_summary = {}

if primary_metric and len(categorical_cols) >= 2:
    for i in range(min(3, len(categorical_cols) - 1)):
        cat1 = categorical_cols[i]
        cat2 = categorical_cols[i + 1]

        segment = (
            df.groupby([cat1, cat2])[primary_metric]
            .sum()
            .sort_values(ascending=False)
            .head(5)
            .round(2)
        )

        segment_summary[f"{primary_metric}_by_{cat1}_and_{cat2}"] = segment.to_dict()

segment_summary

In [ ]:
# Generic charts - This charts the first three useful numeric metrics, excluding ID-like fields.

chart_cols = useful_numeric_cols[:3]

if chart_cols:
    for col in chart_cols:
        df[col].hist(bins=20)

        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.show()
else:
    print("No useful numeric columns found for charting.")

In [ ]:
# Build token-efficient AI prompt

prompt = f"""
You are a senior business data analyst.

Analyze the dataset using ONLY the summarized information below.
Do not invent numbers, columns, or assumptions.

Dataset Structure:
{column_summary}

Primary Metric Selected:
{primary_metric}

Numeric Summary:
{numeric_summary}

Categorical Summary:
{categorical_summary}

Missing Value Summary:
{missing_summary}

Anomaly Summary:
{anomaly_summary}

Relationship Summary:
{relationship_summary}

Concentration Summary:
{concentration_summary}

Correlation Summary:
{correlation_summary}

Segment Summary:
{segment_summary}

Your task:
1. Write a concise executive summary in 3 bullet points
2. Identify 3 deeper insights, not just obvious top values
3. Highlight risks, concentration issues, anomalies, or data quality concerns
4. Recommend 3 follow-up analyses or business actions
5. Clearly state any limitations due to summarized data only

Keep the response professional, concise, and leadership-friendly.
"""

In [ ]:
# Call Open AI

response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt
)

print(response.output_text)

In [ ]:
# Ask a question about your dataset

user_question = input("Ask a question about your dataset: ")

question_prompt = f"""
You are a data analyst.

Use ONLY the summarized dataset information below.
Do not invent numbers, columns, or assumptions.

Dataset Structure:
{column_summary}

Primary Metric Selected:
{primary_metric}

Numeric Summary:
{numeric_summary}

Categorical Summary:
{categorical_summary}

Missing Value Summary:
{missing_summary}

Anomaly Summary:
{anomaly_summary}

Relationship Summary:
{relationship_summary}

Concentration Summary:
{concentration_summary}

Correlation Summary:
{correlation_summary}

Segment Summary:
{segment_summary}

User Question:
{user_question}

Answer the question using only the available summarized data.
If the summarized data is not enough, clearly say what additional analysis or raw columns would be needed.
"""

In [ ]:
# Send your question to OpenAI

response = client.responses.create(
    model="gpt-4.1-mini",
    input=question_prompt
)

print(response.output_text)